# w9_a100.ipynb — BIG jobs (anchor_cap >= 2048, epdb full-population arms)

Job table + labels live in `Pod/w9_jobs.py` (shared with `w9_l40.ipynb`).
Labels are IDENTICAL to the retired `w9_all.ipynb`: claims + result files
interoperate, so pods running any notebook resume/skip each other's work.
This notebook also owns the ONE-TIME full-pool build (150 GB npy).
Run top to bottom; re-run after any interruption — done jobs skip.


In [ ]:
# paths (tree-specific) + flags. Job table / budgets live in Pod/w9_jobs.py.
REPO = os.path.abspath("..")   # this release folder (contains Pod/ and VICReg_review/)
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_FS = "/workspace/w9_out"        # fixed-split results
OUT_CV = "/workspace/w9_cv_out"     # CV results


In [ ]:
# Local setup (release build: the code ships with this folder -- no
# repository synchronisation is needed or performed).
import importlib.util
import os
import sys
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        %pip -q install scikit-learn scipy
        break
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")

In [ ]:
# shared job table + queue machinery (labels compatible with old w9_all runs)
import sys, importlib
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import Pod.w9_jobs
importlib.reload(Pod.w9_jobs)
from Pod import w9_jobs as J
FULL_POOL = J.FULL_POOL
J.summary()


In [ ]:
# Stage the corpus into RAM.
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "wiki_llm_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
# optional prebuilt anchor packs (wscan_gal_rev_g*.npz): stage if present
for s in sorted(src.glob("wscan_gal_rev_g*.npz")):
    d = dst / s.name
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {s.name} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# ONE-TIME (only when FULL_POOL): convert embedding_h5.h5 -> flat fp16 npy +
# meta, written into DATA_SRC on the network volume. ~150 GB, ~20-60 min.
# MULTI-MACHINE SAFE: the builder is claimed atomically; other machines WAIT
# here for the READY marker (this is the one legitimate wait in the campaign).
# The npy is written to a .tmp name and atomically renamed, so a partially
# written pool can never be mistaken for a finished one.
import os, socket, time
from pathlib import Path
import numpy as np

if FULL_POOL:
    H5 = "../game_review_data/embedding_h5.h5"
    dst_v = Path(DATA_SRC) / "full_pool_fp16.npy"
    dst_m = Path(DATA_SRC) / "full_pool_meta.npz"
    ready = Path(DATA_SRC) / "full_pool_READY"
    building = Path(DATA_SRC) / "full_pool_BUILDING"
    me = socket.gethostname() + ":" + os.environ.get("RUNPOD_POD_ID", "?")

    def _claim_build():
        try:
            fd = os.open(building, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.write(fd, me.encode()); os.close(fd)
            return True
        except FileExistsError:
            return building.read_text().strip() == me

    # ADOPT a pool completed by the pre-marker code: open_memmap preallocates
    # the full file, so size proves nothing -- verify content instead (real
    # embeddings are never all-zero; an interrupted build has a zero tail).
    if not ready.exists() and dst_v.exists() and dst_m.exists():
        try:
            v = np.load(dst_v, mmap_mode="r")
            probes = [v.shape[0] - 1, v.shape[0] - 2,
                      v.shape[0] * 3 // 4, v.shape[0] // 2]
            if all(float(np.abs(v[i]).sum()) > 0 for i in probes):
                ready.write_text(f"{me} adopted {v.shape[0]}")
                print("adopted pre-existing COMPLETE pool (content-verified)")
            else:
                print("pre-existing pool is INCOMPLETE (zero tail) -- rebuilding")
            del v
        except Exception as e:
            print("adoption check failed:", e)

    if ready.exists():
        print("full pool already prepared:", dst_v)
    elif _claim_build():
        import h5py
        assert Path(H5).exists(), f"{H5} not found on the volume"
        t0 = time.time()
        tmp_v = dst_v.with_suffix(".npy.tmp")
        with h5py.File(H5, "r") as h:
            N = h["vectors"].shape[0]
            np.savez(dst_m,
                     game_review_offsets=h["game_review_offsets"][:],
                     review_offsets=h["review_offsets"][:],
                     game_names=np.array([g.decode() if isinstance(g, bytes) else str(g)
                                          for g in h["game_names"][:]], object))
            out = np.lib.format.open_memmap(tmp_v, mode="w+", dtype=np.float16,
                                            shape=(N, 1024))
            B = 1_000_000
            for i in range(0, N, B):
                out[i:i+B] = h["vectors"][i:i+B]
                print(f"  {i+min(B, N-i):,}/{N:,} [{time.time()-t0:.0f}s]", flush=True)
            out.flush()
            del out
        os.replace(tmp_v, dst_v)                    # atomic finalize
        ready.write_text(f"{me} {N}")
        building.unlink(missing_ok=True)
        print(f"full pool ready in {(time.time()-t0)/60:.1f} min")
    else:
        owner = building.read_text().strip() if building.exists() else "?"
        print(f"full pool being built by {owner} -- waiting for READY ...", flush=True)
        t0 = time.time()
        while not ready.exists():
            time.sleep(30)
            if int(time.time() - t0) % 300 < 30:
                print(f"  still waiting [{(time.time()-t0)/60:.0f} min]", flush=True)
        print("READY detected; proceeding.")


In [ ]:
# Stage the 150 GB full pool onto FAST LOCAL storage (thread-parallel copy,
# pattern from Pod/h5_staging.py). Network-volume random reads are slow; one
# sequential parallel copy (~5-15 min) buys RAM/NVMe-speed sampling for the
# whole campaign. Falls back to the volume mmap if no local space is found.
import os, sys
from pathlib import Path
FULL_POOL_PATH = ""
if FULL_POOL:
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from Pod.h5_staging import parallel_copy

    src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
    src_m = Path(DATA_SRC) / "full_pool_meta.npz"
    need = src_v.stat().st_size + (5 << 30)

    def _free(p):
        st = os.statvfs(p)
        return st.f_bavail * st.f_frsize

    dest_dir = None
    for cand in ("/dev/shm", "/root/data", "/root"):
        Path(cand).mkdir(parents=True, exist_ok=True)
        if _free(cand) > need:
            dest_dir = Path(cand)
            break
    if dest_dir is None:
        print("WARNING: no local space for the full pool -- workers will mmap "
              "the NETWORK VOLUME copy (slow first pass).")
        FULL_POOL_PATH = str(src_v)
    else:
        dst_v = dest_dir / "full_pool_fp16.npy"
        if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
            print("local full pool already staged:", dst_v)
        else:
            import time
            t0 = time.time()
            tmp = dst_v.with_name(dst_v.name + ".copying")
            print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} "
                  f"(8 threads) ...", flush=True)
            parallel_copy(src_v, tmp, workers=8)
            os.replace(tmp, dst_v)
            print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
        import shutil
        shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
        FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH or "(disabled)")

In [ ]:
# drain the BIG job class. SWEEP_SMALL=True additionally sweeps the other
# class after this one empties (single-pod full-campaign mode).
SWEEP_SMALL = False
fails = J.run_queue("big", repo=REPO, data_dir=DATA_DIR,
                    out_fs=OUT_FS, out_cv=OUT_CV,
                    full_pool_path=FULL_POOL_PATH,
                    sweep_other=SWEEP_SMALL)


In [ ]:
J.aggregate(OUT_FS, OUT_CV)


In [ ]:
# AUTO-STOP removed in the release build: stopping the machine is cloud-
# provider tooling, not part of the experiment. All results are already on
# the shared volume when the run cells finish.
print("run complete -- results are in", OUT_DIR)